# HotpotQA — Raw Data Exploration

## 1. Load dataset

In [41]:
import os
from pathlib import Path

import pandas as pd
from datasets import load_dataset

In [42]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [23]:


ROOT      = Path(".").resolve()
HF_CACHE  = ROOT / "data" / "hotpotqa" / "hf_cache"

os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE)

ds = load_dataset("hotpot_qa", "distractor", cache_dir=str(HF_CACHE))
print(ds)

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})


In [24]:
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()

print(f"Train rows : {len(train_df):,}")
print(f"Validation rows: {len(val_df):,}")
train_df.head(3)

Train rows : 90,447
Validation rows: 7,405


id  \
0  5a7a06935542990198eaf050   
1  5a879ab05542996e4f30887e   
2  5a8d7341554299441c6b9fe5   

                                                                                                                          question  \
0                                                           Which magazine was started first Arthur's Magazine or First for Women?   
1                                                The Oberoi family is part of a hotel company that has a head office in what city?   
2  Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?   

                    answer        type   level  \
0        Arthur's Magazine  comparison  medium   
1                    Delhi      bridge  medium   
2  President Richard Nixon      bridge    hard   

                                                                                              supporting_facts  \
0                                       {'title': ['Arthur's Magazine', 'First for Women'], 'sent_id': [0, 0]}   
1                                          {'title': ['Oberoi family', 'The Oberoi Group'], 'sent_id': [0, 0]}   
2  {'title': ['Allie Goertz', 'Allie Goertz', 'Allie Goertz', 'Milhouse Van Houten'], 'sent_id': [0, 1, 2, 0]}   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## 2. Schema

In [25]:
train_df.dtypes

id                     str
question               str
answer                 str
type                   str
level                  str
supporting_facts    object
context             object
dtype: object

In [26]:
# One full example — shows nested structure
ex = train_df.iloc[0]
print("id             :", ex["id"])
print("question       :", ex["question"])
print("answer         :", ex["answer"])
print("type           :", ex["type"])
print("level          :", ex["level"])
print("supporting_facts:", ex["supporting_facts"])
print()
print("context titles :", ex["context"]["title"])
print("context[0]     :", ex["context"]["sentences"][0])

id             : 5a7a06935542990198eaf050
question       : Which magazine was started first Arthur's Magazine or First for Women?
answer         : Arthur's Magazine
type           : comparison
level          : medium
supporting_facts: {'title': array(["Arthur's Magazine", 'First for Women'], dtype=object), 'sent_id': array([0, 0], dtype=int32)}

context titles : ['Radio City (Indian radio station)' 'History of Albanian football'
 'Echosmith' "Women's colleges in the Southern United States"
 'First Arthur County Courthouse and Jail' "Arthur's Magazine"
 '2014–15 Ukrainian Hockey Championship' 'First for Women'
 'Freeway Complex Fire' 'William Rast']
context[0]     : ["Radio City is India's first private FM radio station and was started on 3 July 2001."
 ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).'
 ' It plays Hindi, English and regional songs.'
 ' It was

## 3. Basic statistics

In [27]:
print("=== Question type distribution ===")
print(train_df["type"].value_counts())
print()
print("=== Difficulty level distribution ===")
print(train_df["level"].value_counts())

=== Question type distribution ===
type
bridge        72991
comparison    17456
Name: count, dtype: int64

=== Difficulty level distribution ===
level
medium    56814
easy      17972
hard      15661
Name: count, dtype: int64


In [28]:
train_df["q_len"] = train_df["question"].str.len()
train_df["a_len"] = train_df["answer"].str.len()

print("=== Question length (chars) ===")
print(train_df["q_len"].describe().round(1))
print()
print("=== Answer length (chars) ===")
print(train_df["a_len"].describe().round(1))

=== Question length (chars) ===
count    90447.0
mean       105.6
std         59.1
min         13.0
25%         69.0
50%         90.0
75%        120.0
max        654.0
Name: q_len, dtype: float64

=== Answer length (chars) ===
count    90447.0
mean        13.7
std         11.8
min          1.0
25%          7.0
50%         12.0
75%         17.0
max        623.0
Name: a_len, dtype: float64


In [29]:
# Number of supporting facts per question
train_df["n_supporting"] = train_df["supporting_facts"].apply(lambda x: len(x["title"]))
print("=== Supporting facts count ===")
print(train_df["n_supporting"].value_counts().sort_index())

=== Supporting facts count ===
n_supporting
2     63676
3     20017
4      5814
5       724
6       141
7        52
8        17
9         4
11        1
12        1
Name: count, dtype: int64


In [30]:
# Number of context articles per question (always 10 in distractor setting)
train_df["n_context"] = train_df["context"].apply(lambda x: len(x["title"]))
print("=== Context articles per question ===")
print(train_df["n_context"].value_counts())

=== Context articles per question ===
n_context
10    89609
2       262
3       156
4        94
5        88
7        77
8        60
6        53
9        48
Name: count, dtype: int64


## 4. Flatten context into passage-level rows

In [31]:
records = []
for _, row in train_df.iterrows():
    for title, sents in zip(row["context"]["title"], row["context"]["sentences"]):
        records.append({
            "qid":   row["id"],
            "title": title,
            "text":  " ".join(sents),
        })

passages_df = pd.DataFrame(records)
print(f"Total passages: {len(passages_df):,}")
passages_df.head(5)

Total passages: 899,667


,qid,title,text
0,5a7a06935542990198eaf050,Radio City (Indian radio station),"Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company."
1,5a7a06935542990198eaf050,History of Albanian football,"Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tournament during 1929-1931, which started in 1929 (although Albania eventually had pressure from the teams because of competition, competition started first and was strong enough in the duels) . Albanian National Team was founded on June 6, 1930, but Albania had to wait 16 years to play its first international match and then defeated Yugoslavia in 1946. In 1932, Albania joined FIFA (during the 12–16 June convention ) And in 1954 she was one of the founding members of UEFA."
2,5a7a06935542990198eaf050,Echosmith,"Echosmith is an American, Corporate indie pop band formed in February 2009 in Chino, California. Originally formed as a quartet of siblings, the band currently consists of Sydney, Noah and Graham Sierota, following the departure of eldest sibling Jamie in late 2016. Echosmith started first as ""Ready Set Go!"" until they signed to Warner Bros. Records in May 2012. They are best known for their hit song ""Cool Kids"", which reached number 13 on the ""Billboard"" Hot 100 and was certified double platinum by the RIAA with over 1,200,000 sales in the United States and also double platinum by ARIA in Australia. The song was Warner Bros. Records' fifth-biggest-selling-digital song of 2014, with 1.3 million downloads sold. The band's debut album, ""Talking Dreams"", was released on October 8, 2013."
3,5a7a06935542990198eaf050,Women's colleges in the Southern United States,"Women's colleges in the Southern United States refers to undergraduate, bachelor's degree–granting institutions, often liberal arts colleges, whose student populations consist exclusively or almost exclusively of women, located in the Southern United States. Many started first as girls' seminaries or academies. Salem College is the oldest female educational institution in the South and Wesleyan College is the first that was established specifically as a college for women. Some schools, such as Mary Baldwin University and Salem College, offer coeducational courses at the graduate level."
4,5a7a06935542990198eaf050,First Arthur County Courthouse and Jail,"The First Arthur County Courthouse and Jail, was perhaps the smallest court house in the United States, and serves now as a museum."


In [32]:
passages_df["text_len"] = passages_df["text"].str.len()
print("=== Passage text length (chars) ===")
print(passages_df["text_len"].describe().round(1))

=== Passage text length (chars) ===
count    899667.0
mean        546.1
std         336.0
min          24.0
25%         324.0
50%         483.0
75%         689.0
max        8268.0
Name: text_len, dtype: float64


## 5. Sample questions by type

In [33]:
print("=== Comparison questions (sample) ===")
sample = train_df[train_df["type"] == "comparison"][["question", "answer"]].sample(5, random_state=42)
for _, r in sample.iterrows():
    print(f"  Q: {r['question']}")
    print(f"  A: {r['answer']}")
    print()

=== Comparison questions (sample) ===
  Q: Were Theodore H. White and Hilaire Belloc both historians ?
  A: yes

  Q: Are Ralph Bakshi and Herschell Gordon Lewis both American?
  A: yes

  Q: What is from farther west, Killdozer or Skinny Puppy?
  A: Skinny Puppy

  Q: Which industry do Richard Hawley and Chicago's Catherine belong to? 
  A: rock band

  Q: Ares and Right On!, is which type of publication?
  A: magazine



In [34]:
print("=== Bridge questions (sample) ===")
sample = train_df[train_df["type"] == "bridge"][["question", "answer"]].sample(5, random_state=42)
for _, r in sample.iterrows():
    print(f"  Q: {r['question']}")
    print(f"  A: {r['answer']}")
    print()

=== Bridge questions (sample) ===
  Q: Droga5 has completed advertisement campaigns for what founder and Chief Creative Officer of Ecko Unlimited?
  A: Marc Louis "Eckō" Milecofsky

  Q: When was the American baseball infielder, manager, and coach died who was hired by Emil Fuchs to replace him?
  A: January 5, 1963

  Q: Who distributed the film which featured a boy who had previously had no film experience?
  A: Warner Bros. Pictures

  Q: What is the place popular for which is in between Fair Harbor in Suffolk County and a village on Fire Island in the southern part of the Town of Islip in Suffolk County?
  A: kayaking

  Q: Who designed the theater that commissioned Benjamin Britten's opera?
  A: Joseph Bové



## 6. Validation set

In [35]:
print("=== Validation type distribution ===")
print(val_df["type"].value_counts())
print()
print("=== Validation level distribution ===")
print(val_df["level"].value_counts())
val_df[["question", "answer", "type", "level"]].head(5)

=== Validation type distribution ===
type
bridge        5918
comparison    1487
Name: count, dtype: int64

=== Validation level distribution ===
level
hard    7405
Name: count, dtype: int64


,question,answer,type,level
0,Were Scott Derrickson and Ed Wood of the same nationality?,yes,comparison,hard
1,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,bridge,hard
2,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,bridge,hard
3,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,comparison,hard
4,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City",bridge,hard


In [38]:
val_df[['question', 'answer']].head(20)

,question,answer
0,Were Scott Derrickson and Ed Wood of the same nationality?,yes
1,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol
2,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs
3,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no
4,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City"
5,2014 S/S is the debut album of a South Korean boy group that was formed by who?,YG Entertainment
6,Who was known by his stage name Aladin and helped organizations improve their performance as a consultant?,Eenasul Fateh
7,The arena where the Lewiston Maineiacs played their home games can seat how many people?,"3,677 seated"
8,"Who is older, Annie Morton or Terry Richardson?",Terry Richardson
9,Are Local H and For Against both from the United States?,yes


### EVALUATION ANALYSIS

In [43]:
eval_df = pd.read_csv('./results/eval_20260515_160704.csv')

In [44]:
eval_df

,id,question,real_answer,llm_answer
0,5ae143ed55429920d5234360,In what year was the university where Sergei Aleksandrovich Tokarev was a professor founded?,1755,I cannot find the answer in the provided context.
1,5abc19705542993a06baf86e,Black Book starred the actress and writer of what heritage?,Dutch,Dutch
2,5ac3e0f7554299194317388b,Which actor does American Beauty and American Beauty have in common?,Kevin Spacey,Kevin Spacey
3,5ae518655542993aec5ec139,Ken Pruitt was a Republican member of an upper house of the legislature with how many members ?,40 members,40
4,5ab985eb554299131ca42360,"Between Greyia and Calibanus, which genus contains more species?",Greyia,yes
5,5a8753d95542994846c1cd63,Did John Updike and Tom Clancy both publish more than 15 bestselling novels?,yes,yes
6,5ab97d0a5542996be202051e,Who was hung for assisting the attempted surrender of a defector from the American Continental Army to the British Army?,John André,John André
7,5adef1b35542993a75d263af,which Mexican and American film actress is Ethel Houbiers French voice of,Salma Hayek Pinault,Salma Hayek
8,5a73b55855429978a71e9086,Which major international airport in south-east England ranks as the 8th busiest airport in Europe and replaced Croydon Airport?,Gatwick Airport,Gatwick
9,5ae80919554299540e5a56f6,Isabella Kelly was born at a ruined castle characterized as one of the most isolated fortifications in Britain by who?,The Changing Scottish Landscape,I cannot find the answer in the provided context.


In [46]:
eval_metrics = pd.read_json('./results/smoke.json')

ValueError: Trailing data